# Inspect ResNet-18 Weights Before and After PTQ

This notebook compares the kernel weights in every Conv2D and Dense layer of the trained CIFAR-10 ResNet-18:

1. Original FP32 Keras weights.
2. Actual INT8 constant weights stored in the built-in full-integer TFLite model.
3. Actual INT8 arrays produced by the custom per-output-channel quantizer.

Important: TFLite folds BatchNorm into preceding convolution kernels and stores Conv2D weights as `OHWI`. Custom PTQ currently quantizes the original, unfolded Keras kernels in `HWIO`. This notebook converts TFLite arrays back to Keras layout for inspection, but their numerical values can still differ because of BatchNorm folding.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.quantization.custom_quantization import custom_ptq

## 1. Load the trained and built-in PTQ models

Run notebooks 06 and 07 first if either artifact is missing.

In [ ]:
FP32_MODEL_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "resnet18_cifar10_training"
    / "resnet18_cifar10_fp32.keras"
)
BUILTIN_MODEL_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "resnet18_ptq_accuracy_notebook"
    / "resnet18_cifar10_builtin_full_int8.tflite"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "resnet18_weight_inspection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for required_path in (FP32_MODEL_PATH, BUILTIN_MODEL_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing artifact: {required_path}. Run notebooks 06 and 07 first."
        )

model = keras.models.load_model(FP32_MODEL_PATH, compile=False)
interpreter = tf.lite.Interpreter(model_path=str(BUILTIN_MODEL_PATH))
interpreter.allocate_tensors()

print(f"FP32 model: {FP32_MODEL_PATH}")
print(f"Built-in INT8 model: {BUILTIN_MODEL_PATH}")

## 2. Collect original FP32 kernels

The rank condition means tensor rank, not weight magnitude. Rank-4 Conv2D kernels and rank-2 Dense kernels are included; rank-1 biases and BatchNorm parameters are listed separately later.

In [ ]:
fp32_kernels = {}
excluded_rank1 = []

for weight in model.weights:
    values = weight.numpy()
    path = getattr(weight, "path", weight.name)
    if np.issubdtype(values.dtype, np.floating) and values.ndim >= 2:
        layer_name = path.rsplit("/", 1)[0]
        fp32_kernels[layer_name] = {
            "path": path,
            "values": values,
            "layer_type": "Conv2D" if values.ndim == 4 else "Dense",
        }
    elif np.issubdtype(values.dtype, np.floating):
        excluded_rank1.append({
            "path": path,
            "shape": tuple(values.shape),
            "reason": "rank-1 bias or BatchNorm parameter",
        })

print(f"FP32 Conv2D/Dense kernels: {len(fp32_kernels)}")
print(f"FP32 rank-1 tensors excluded by custom PTQ: {len(excluded_rank1)}")

## 3. Extract actual built-in INT8 weights from TFLite

Each TFLite Conv2D/FullyConnected operator identifies its constant weight input. Operator output names are used to recover the original Keras layer name. The stored arrays are then transposed from `OHWI → HWIO` for Conv2D and `OI → IO` for Dense.

In [ ]:
tensor_details = {
    int(detail["index"]): detail for detail in interpreter.get_tensor_details()
}
builtin_kernels = {}

for operation in interpreter._get_ops_details():
    if operation["op_name"] not in {"CONV_2D", "FULLY_CONNECTED"}:
        continue

    weight_index = int(operation["inputs"][1])
    output_index = int(operation["outputs"][0])
    weight_detail = tensor_details[weight_index]
    output_name = tensor_details[output_index]["name"]
    stored_values = interpreter.get_tensor(weight_index).copy()
    quantization = weight_detail["quantization_parameters"]

    if operation["op_name"] == "CONV_2D":
        matches = re.findall(r"/([A-Za-z0-9_]+)_1/convolution", output_name)
        if not matches:
            raise ValueError(f"Could not identify Conv2D layer from: {output_name}")
        layer_name = matches[-1]
        keras_layout_values = np.transpose(stored_values, (1, 2, 3, 0))
    else:
        layer_name = "cifar10_predictions"
        keras_layout_values = stored_values.T

    scales = quantization["scales"].astype(np.float32)
    zero_points = quantization["zero_points"].astype(np.int32)
    dequantized = (
        keras_layout_values.astype(np.float32)
        - zero_points.reshape((1,) * (keras_layout_values.ndim - 1) + (-1,))
    ) * scales.reshape((1,) * (keras_layout_values.ndim - 1) + (-1,))

    builtin_kernels[layer_name] = {
        "values": keras_layout_values,
        "stored_values": stored_values,
        "stored_shape": tuple(stored_values.shape),
        "scales": scales,
        "zero_points": zero_points,
        "dequantized": dequantized,
        "tflite_tensor_name": weight_detail["name"],
    }

missing_builtin = sorted(set(fp32_kernels) - set(builtin_kernels))
if missing_builtin:
    raise ValueError(f"Built-in weights missing for layers: {missing_builtin}")
print(f"Built-in INT8 kernels extracted: {len(builtin_kernels)}")

## 4. Collect actual custom INT8 weights

In [ ]:
custom_result = custom_ptq(
    model, representative_samples=None, quantize_min_rank=2, per_channel=True
)["weights"]
custom_kernels = {}

kernel_items = list(fp32_kernels.items())
if len(kernel_items) != len(custom_result.tensors):
    raise ValueError(
        f"FP32/custom kernel count mismatch: {len(kernel_items)} vs "
        f"{len(custom_result.tensors)}"
    )

for (layer_name, fp32_info), quantized_tensor in zip(
    kernel_items, custom_result.tensors
):
    scales = np.asarray(quantized_tensor.scale, dtype=np.float32)
    scale_shape = (1,) * (quantized_tensor.values.ndim - 1) + (-1,)
    custom_kernels[layer_name] = {
        "values": quantized_tensor.values.copy(),
        "scales": scales,
        "zero_points": np.zeros(scales.shape, dtype=np.int32),
        "dequantized": (
            quantized_tensor.values.astype(np.float32)
            * scales.reshape(scale_shape)
        ),
    }

print(f"Custom INT8 kernels collected: {len(custom_kernels)}")

## 5. Layer-by-layer summary

This table covers every quantized Conv2D and Dense kernel. The built-in scale and weight values reflect BatchNorm folding for convolution layers; custom values do not.

In [ ]:
summary_rows = []
layer_weights = {}

for layer_name, fp32_info in fp32_kernels.items():
    fp32_values = fp32_info["values"]
    builtin_info = builtin_kernels[layer_name]
    custom_info = custom_kernels[layer_name]
    if fp32_values.shape != builtin_info["values"].shape:
        raise ValueError(f"Built-in shape mismatch for {layer_name}.")
    if fp32_values.shape != custom_info["values"].shape:
        raise ValueError(f"Custom shape mismatch for {layer_name}.")

    layer_weights[layer_name] = {
        "fp32": fp32_values,
        "builtin_int8": builtin_info["values"],
        "builtin_dequantized": builtin_info["dequantized"],
        "builtin_scales": builtin_info["scales"],
        "custom_int8": custom_info["values"],
        "custom_dequantized": custom_info["dequantized"],
        "custom_scales": custom_info["scales"],
    }
    summary_rows.append({
        "layer": layer_name,
        "type": fp32_info["layer_type"],
        "keras_shape": tuple(fp32_values.shape),
        "tflite_stored_shape": builtin_info["stored_shape"],
        "weight_count": fp32_values.size,
        "fp32_min": float(fp32_values.min()),
        "fp32_max": float(fp32_values.max()),
        "builtin_int8_min": int(builtin_info["values"].min()),
        "builtin_int8_max": int(builtin_info["values"].max()),
        "builtin_scale_count": builtin_info["scales"].size,
        "custom_int8_min": int(custom_info["values"].min()),
        "custom_int8_max": int(custom_info["values"].max()),
        "custom_scale_count": custom_info["scales"].size,
        "custom_round_trip_mae": float(
            np.mean(np.abs(fp32_values - custom_info["dequantized"]))
        ),
        "builtin_batchnorm_folded": fp32_info["layer_type"] == "Conv2D",
    })

layer_summary = pd.DataFrame(summary_rows)
layer_summary.to_csv(OUTPUT_DIR / "layer_weight_summary.csv", index=False)
layer_summary

## 6. Inspect the weights of any layer

Change `LAYER_TO_INSPECT` to any name from `layer_summary['layer']`. The preview uses the same flattened Keras ordering for all three arrays. Increase `NUMBER_OF_VALUES` as needed. The complete NumPy arrays remain available in `layer_weights[LAYER_TO_INSPECT]`.

In [ ]:
layer_summary[["layer", "type", "keras_shape"]]

In [ ]:
LAYER_TO_INSPECT = "conv1_conv"
NUMBER_OF_VALUES = 32

if LAYER_TO_INSPECT not in layer_weights:
    raise KeyError(
        f"Unknown layer {LAYER_TO_INSPECT!r}. Choose from {list(layer_weights)}"
    )

selected = layer_weights[LAYER_TO_INSPECT]
value_count = min(NUMBER_OF_VALUES, selected["fp32"].size)
weight_preview = pd.DataFrame({
    "flat_index": np.arange(value_count),
    "fp32_before": selected["fp32"].reshape(-1)[:value_count],
    "builtin_int8_after": selected["builtin_int8"].reshape(-1)[:value_count],
    "builtin_dequantized": (
        selected["builtin_dequantized"].reshape(-1)[:value_count]
    ),
    "custom_int8_after": selected["custom_int8"].reshape(-1)[:value_count],
    "custom_dequantized": (
        selected["custom_dequantized"].reshape(-1)[:value_count]
    ),
})

print(f"Layer: {LAYER_TO_INSPECT}")
print(f"Shape in Keras layout: {selected['fp32'].shape}")
print(f"Built-in scales: {selected['builtin_scales'].shape}")
print(f"Custom scales: {selected['custom_scales'].shape}")
weight_preview

## 7. Inspect per-output-channel scales for the selected layer

In [ ]:
scale_count = min(32, selected["custom_scales"].size)
scale_preview = pd.DataFrame({
    "output_channel": np.arange(scale_count),
    "builtin_scale_after_bn_folding": (
        selected["builtin_scales"][:scale_count]
    ),
    "custom_scale_original_kernel": (
        selected["custom_scales"][:scale_count]
    ),
})
scale_preview

## 8. Tensors not quantized by the custom weight path

These rank-1 tensors are deliberately excluded. Built-in full-integer conversion folds BatchNorm into convolution weights and represents applicable biases using higher-precision integer constants.

In [ ]:
excluded_table = pd.DataFrame(excluded_rank1)
excluded_table.to_csv(OUTPUT_DIR / "custom_ptq_excluded_rank1.csv", index=False)
excluded_table